# Run a linear regression on contralateral direction for STOP cue and saccade onset

## Load packages and prepare data

In [2]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD, PCA
from pprint import pprint

import os
os.chdir('..')
import multi_session_pca
importlib.reload(multi_session_pca)
from multi_session_pca import MultiSessionPCA

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib widget

In [5]:
analyzer = MultiSessionPCA({
    'alignment_point': 'stop_cue',
    'epok': [0, 400],
    'bin_size': 1,
    'smooth_ker_size': 15,
    'ssd_number': None,
    'normalize': False,
    'delta': False,
    'subtract_average_PSTH': False
})

# Check default config
print("Default Configuration:")
pprint(analyzer.config)

Default Configuration:
{'alignment_point': 'go_cue',
 'bin_size': 1,
 'delta': False,
 'epok': [-100, 200],
 'excluded_sessions': ['fi210628', 'fi210629', 'fi210704'],
 'figsize_2d_grid': (7, 19),
 'figsize_3d': (12, 10),
 'figsize_time': (10, 3),
 'min_cells_per_session': 10,
 'min_trials_per_condition': 5,
 'monkey': 'fiona',
 'n_pca_components': 5,
 'n_workers': 18,
 'normalize': False,
 'required_directions': [0, 180],
 'required_trial_types': ['GO', 'STOP', 'CONT'],
 'smooth_ker_size': 15,
 'ssd_number': None,
 'subtract_average_PSTH': False}


In [7]:
analyzer.load_data('../data/unified_cell_trial_data/alt_saccade_params_msn_fiona_cell_trial_data_with_slow_go_data.pkl')
# analyzer.cell_df = analyzer.cell_df[analyzer.cell_df['cell_type'] == 'msn']
# analyzer.cell_df = analyzer.cell_df[analyzer.cell_df['trial_session'] == 'fi211025a']

analyzer.validate_all_sessions()
print(f"✓ Valid sessions: {len(analyzer.valid_sessions)}")

cell_df = analyzer.cell_df[analyzer.cell_df['trial_session'].isin(analyzer.valid_sessions)]
cell_df = cell_df[~cell_df['cell_ID'].isin([2134])]
cell_df = cell_df[~cell_df['cell_ID'].isin([1352])]

Loading data from: ../data/unified_cell_trial_data/alt_saccade_params_msn_fiona_cell_trial_data_with_slow_go_data.pkl
✓ Database loaded: 1,939,269 cell-trial combinations
  Total sessions: 61
  Total unique cells: 3096
✓ Session statistics computed for 61 sessions
  Excluded sessions: 1
  Candidate sessions: 60
Validating sessions...
--------------------------------------------------------------------------------
Cell 2134 removed from session fi211110a.
Dropped 1 cell with incomplete trial type or directional data.
Cell 1352 removed from session fi211018a.
Dropped 1 cell with incomplete trial type or directional data.
✓ fi211102a: Valid (189 cells)
✓ fi211025a: Valid (155 cells)
✓ fi211110a: Valid (145 cells)
✓ fi211122a: Valid (132 cells)
✓ fi211109a: Valid (132 cells)
✓ fi211111a: Valid (130 cells)
✓ fi211104a: Valid (121 cells)
✓ fi211026a: Valid (116 cells)
✓ fi211115a: Valid (111 cells)
✓ fi211108a: Valid (98 cells)
✓ fi211118a: Valid (95 cells)
✓ fi211018a: Valid (94 cells)
✓ fi

In [8]:
# Load data
# cell_df = pd.read_pickle('../../data/unified_cell_trial_data/msn_fiona_cell_trial_data.pkl')
# cell_df = pd.read_pickle('../../data/unified_cell_trial_data/alt_saccade_params_msn_fiona_cell_trial_data.pkl')

# Filter data
# excluded_sessions = ['fi210628', 'fi210629', 'fi210704']
# cell_df = cell_df[~cell_df['trial_session'].isin(excluded_sessions)]
# cell_df = cell_df[cell_df['trial_failed'] == False]

# Filter to sessions with enough cells
min_cells = 15
session_cell_counts = cell_df.groupby('trial_session')['cell_ID'].nunique()
valid_sessions = session_cell_counts[session_cell_counts >= min_cells].index.tolist()
cell_df = cell_df[cell_df['trial_session'].isin(valid_sessions)]

print(f"Data loaded: {len(cell_df):,} trials, {cell_df['cell_ID'].nunique()} cells")

Data loaded: 1,933,976 trials, 3073 cells


## Set regression labels and regressors

$y_i = \beta_0 + \beta_{go}x_1 + \beta_{stop}x_2$
Where $i$ indicates the cell ID and $x_i \in {0,1}$
